# 02 — Classification: predicting a binary outcome

We estimate whether a customer churns using generated data. This is a lesson about modeling and evaluation, not a claim about actual customers.

## 1. Bernoulli likelihood and convex optimization

With linear predictor $\eta_i=\beta_0+x_i^T\beta$ and $p_i=\sigma(\eta_i)$, Bernoulli likelihood is $p_i^{y_i}(1-p_i)^{1-y_i}$. The negative log likelihood per observation is

$$\ell_i(\beta)=\log(1+e^{\eta_i})-y_i\eta_i,$$

and empirical risk is $L(\beta)=n^{-1}\sum_i\ell_i(\beta)$. Differentiating through $p_i=\sigma(\eta_i)$ gives slope gradient $X^T(p-y)/n$ and Hessian $X^TWX/n$, where $W_{ii}=p_i(1-p_i)\ge0$. The Hessian is positive semidefinite, so the logistic loss is convex. Adding $\lambda\|\beta\|_2^2/2$ adds $\lambda\beta$ to the gradient and $\lambda I$ to the slope Hessian block. A positive penalty makes that block positive definite. The intercept is conventionally unpenalized; scikit-learn handles it separately.

## 2. Load synthetic data

Rows were generated for instruction. They do not represent real customers, prevalence, or intervention effects.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_data_file(filename):
    """Search the current directory and its parents for a repository data file."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        data_path = directory / "data" / filename
        if data_path.is_file():
            return data_path
    raise FileNotFoundError(
        f"Could not find data/{filename}. Start Jupyter at the repository root "
        "or inside this lecture directory."
    )


churn_path = find_data_file("churn.csv")
customer_data = pd.read_csv(churn_path)
print(f"Loaded {len(customer_data)} synthetic rows from {churn_path}")
customer_data.head()

## 3. Training, validation, and final test

We reserve 20% of observations as a final test set. The remaining development data are split into training and validation partitions. Stratification keeps class proportions similar. Cross-validation and regularization selection use training rows; validation selects a threshold under a stated toy cost; the final test set evaluates the frozen model and threshold once.

In [ ]:
from sklearn.model_selection import train_test_split

feature_data = customer_data.drop(columns="churn")
churn_response = customer_data["churn"].astype(int)

(
    development_features,
    final_test_features,
    development_response,
    final_test_response,
) = train_test_split(
    feature_data,
    churn_response,
    test_size=0.20,
    random_state=19,
    stratify=churn_response,
)
(
    training_features,
    validation_features,
    training_response,
    validation_response,
) = train_test_split(
    development_features,
    development_response,
    test_size=0.25,
    random_state=20,
    stratify=development_response,
)

print(f"Training rows: {len(training_features)}")
print(f"Validation rows: {len(validation_features)}")
print(f"Final test rows: {len(final_test_features)}")
print("Training class proportions:")
training_response.value_counts(normalize=True).sort_index()

## 4. Pipeline for mixed inputs

Scale numerical features and one-hot encode contract type. Encoding categories as integers would suggest an artificial order. All transformations are part of the pipeline, so cross-validation refits them within each fold.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = ["tenure_months", "monthly_charge", "support_calls"]
categorical_columns = ["contract"]

numeric_preprocessing = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_preprocessing = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessing = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessing, numeric_columns),
        ("category", categorical_preprocessing, categorical_columns),
    ]
)
classification_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)

## 5. Regularization on training rows

The derivation above writes a penalty $\lambda\|\beta\|^2/2$. scikit-learn exposes inverse strength $C$: smaller $C$ means stronger regularization, though the exact scaling convention depends on sample size and solver. We select $C$ with stratified folds inside the training partition. Validation and final test data do not participate in this selection.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

training_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=19)
regularization_search = GridSearchCV(
    estimator=classification_pipeline,
    param_grid={"model__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]},
    scoring="roc_auc",
    cv=training_folds,
    refit=True,
)
regularization_search.fit(training_features, training_response)

selected_inverse_penalty = regularization_search.best_params_["model__C"]
print(f"Selected C: {selected_inverse_penalty}")
print(f"Mean training-fold ROC-AUC: {regularization_search.best_score_:.3f}")

## 6. Validation metrics before choosing a threshold

At threshold 0.5, accuracy is the fraction of correct class decisions. Precision is the share of predicted churners who actually churn; recall is the share of churners found. Their harmonic mean is $F_1=2PR/(P+R)$. ROC-AUC is the probability that a randomly selected positive receives a higher score than a randomly selected negative, counting ties as one half. It assesses ranking, not probability calibration or a particular policy.

The default 0.5 cutoff is a reference, not a universal policy. This section reports validation metrics at 0.5 and a majority-class baseline. The final test partition remains untouched.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

best_classifier = regularization_search.best_estimator_
validation_probabilities = best_classifier.predict_proba(validation_features)[:, 1]
validation_predictions_at_half = (validation_probabilities >= 0.5).astype(int)

majority_baseline = DummyClassifier(strategy="most_frequent")
majority_baseline.fit(training_features, training_response)
baseline_predictions = majority_baseline.predict(validation_features)

print("Validation majority-baseline accuracy:",
      round(accuracy_score(validation_response, baseline_predictions), 3))
print("Validation metrics at threshold 0.5:", {
    "accuracy": round(accuracy_score(validation_response, validation_predictions_at_half), 3),
    "precision": round(precision_score(validation_response, validation_predictions_at_half, zero_division=0), 3),
    "recall": round(recall_score(validation_response, validation_predictions_at_half, zero_division=0), 3),
    "F1": round(f1_score(validation_response, validation_predictions_at_half, zero_division=0), 3),
    "ROC-AUC": round(roc_auc_score(validation_response, validation_probabilities), 3),
})
print("Validation confusion matrix [actual rows, predicted columns]:\n",
      confusion_matrix(validation_response, validation_predictions_at_half))

Accuracy alone can hide failure to identify the positive class. A majority baseline can score well if non-churn is common while detecting no churners. Compare precision and recall with the actual use case.

## 7. Choose a threshold on validation, then test once

Suppose a false positive has cost $C_{FP}$ and a false negative has cost $C_{FN}$; correct decisions have zero cost. With calibrated probability $p$, predicting churn costs $C_{FP}(1-p)$ in expectation, while predicting stay costs $C_{FN}p$. Choose churn when

$$C_{FP}(1-p)\le C_{FN}p\quad\Longleftrightarrow\quad p\ge\frac{C_{FP}}{C_{FP}+C_{FN}}.$$

This threshold assumes the probabilities are calibrated for the decision population, costs are known and constant, and decisions have no shared capacity constraint. Here we use a toy ratio $C_{FP}=1$, $C_{FN}=5$, giving a theoretical cutoff $1/6\approx0.167$. We inspect several cutoffs on validation data to see the finite-sample tradeoff and select the lowest estimated cost. Then we freeze the threshold and evaluate the held-out test set once. The exercise's costs are for mathematics practice only.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(validation_response, validation_probabilities)
plt.title("Validation ROC curve")
plt.show()

false_positive_cost = 1.0
false_negative_cost = 5.0
threshold_candidates = [0.10, 0.17, 0.25, 0.35, 0.50, 0.65]
threshold_results = []

for threshold in threshold_candidates:
    validation_decisions = (validation_probabilities >= threshold).astype(int)
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        validation_response, validation_decisions, labels=[0, 1]
    ).ravel()
    estimated_cost = (false_positive_cost * false_positive
                      + false_negative_cost * false_negative)
    threshold_results.append(
        {
            "threshold": threshold,
            "flagged": int(validation_decisions.sum()),
            "false_positives": false_positive,
            "false_negatives": false_negative,
            "estimated_cost": estimated_cost,
        }
    )

validation_cost_table = pd.DataFrame(threshold_results)
validation_cost_table

The validation table is sorted by the toy estimated cost. We select the threshold from validation and keep it fixed. The final test data will be evaluated once with that threshold.

In [ ]:
validation_cost_table = validation_cost_table.sort_values(
    ["estimated_cost", "threshold"]
)
selected_threshold = float(validation_cost_table.iloc[0]["threshold"])

final_test_probabilities = best_classifier.predict_proba(final_test_features)[:, 1]
final_test_predictions = (final_test_probabilities >= selected_threshold).astype(int)

print(f"Chosen validation threshold: {selected_threshold:.2f}")
print("Final test metrics:", {
    "accuracy": round(accuracy_score(final_test_response, final_test_predictions), 3),
    "precision": round(precision_score(final_test_response, final_test_predictions, zero_division=0), 3),
    "recall": round(recall_score(final_test_response, final_test_predictions, zero_division=0), 3),
    "F1": round(f1_score(final_test_response, final_test_predictions, zero_division=0), 3),
    "ROC-AUC": round(roc_auc_score(final_test_response, final_test_probabilities), 3),
})
print("Final test confusion matrix [actual rows, predicted columns]:\n",
      confusion_matrix(final_test_response, final_test_predictions))

## 8. Coefficients as conditional log-odds associations

A coefficient changes fitted log odds by its value for a one-unit increase, or a one-standard-deviation increase for a scaled numerical feature, conditional on the other encoded inputs. Exponentiating gives a model odds ratio. Odds ratios are not risk ratios, and observational coefficients do not establish causal effects.

In [ ]:
encoded_feature_names = best_classifier.named_steps["preprocessing"].get_feature_names_out()
log_odds_coefficients = pd.Series(
    best_classifier.named_steps["model"].coef_[0],
    index=encoded_feature_names,
)
log_odds_coefficients.reindex(
    log_odds_coefficients.abs().sort_values(ascending=False).index
).head(10).round(3)

## Takeaways

- Train on one partition, select regularization with training-only cross-validation, and select a decision threshold on validation data.
- Evaluate the frozen model and threshold once on final test data.
- Report class counts, confusion matrix, and metrics relevant to the intended decision.
- Generated-data performance does not establish real customer outcomes or causal effects.